In [2]:
##################
### Keagan Moo ###
### 2025/10/01 ###
### ATAC dbGap ###
##################

# This script loads in and filters the ATAC and RNA total cross cell type counts matricies for the FUSION
# Skeletal Muscle data for samples that consented to data sharing. Subsequently the sample IDs are anonymized
# according to the common reference at:
# /nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/dbgap/molecular/sn_muscle_nucleus_cluster_sample_mapping.tsv

suppressPackageStartupMessages({library(SingleCellExperiment)
library(Matrix)
library(dplyr)
library(ggplot2)
library(glue)
library(MAST)
})

In [2]:
#ATAC First
cDex <- "Adipocyte"
small <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/{cDex}_FUSIONsnATAC_sce.rds"))
keepRows <- rowSums(counts(small) >= 1) >= ncol(counts(small))*0.1
small <- small[keepRows,]

In [3]:
dim(counts(small))
head(colData(small))

[1] 38409 42804

DataFrame with 6 rows and 21 columns
                                                index     SNG.1ST    UMAP_1
                                          <character> <character> <numeric>
atac.499.NM.1.AAACTCGAGTATACGA atac.499.NM.1.AAACTC..       32048  -3.53715
atac.499.NM.1.AAACTGCCACAGGTAG atac.499.NM.1.AAACTG..       12094  -5.70362
atac.499.NM.1.AAAGGATGTATGCTAC atac.499.NM.1.AAAGGA..       32001  -4.91979
atac.499.NM.1.AAAGGGCAGTTCCAGT atac.499.NM.1.AAAGGG..       12011  -2.00075
atac.499.NM.1.AAAGGGCGTTAGGAGC atac.499.NM.1.AAAGGG..       12136  -4.98213
atac.499.NM.1.AAATGAGCAATGACTC atac.499.NM.1.AAATGA..       32052  -3.00428
                                  UMAP_2    modality     library
                               <numeric> <character> <character>
atac.499.NM.1.AAACTCGAGTATACGA -0.582529        atac    499-NM-1
atac.499.NM.1.AAACTGCCACAGGTAG  0.420222        atac    499-NM-1
atac.499.NM.1.AAAGGATGTATGCTAC -0.588620        atac    499-NM-1
atac.499.NM.1.AAAGGGCAGTTCCAGT

In [3]:
reFrame <- read.table("/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/dbgap/molecular/sn_muscle_nucleus_cluster_sample_mapping.tsv", sep = "\t", header = TRUE)

In [ ]:
dim(reFrame[reFrame$modality == "atac",])

head(reFrame)
(unique(reFrame$coarse_cluster_name))

In [ ]:
indexList <- reFrame[(reFrame$modality == "atac") & (reFrame$coarse_cluster_name == glue("{cDex}")),]$index
length(indexList)
head(small$index)
head(indexList)
length(small$index %in% indexList)

In [ ]:
smallFiltered <- small[,small$index %in% indexList]
dim(counts(smallFiltered))

In [ ]:
head(data.frame(as.matrix(counts(smallFiltered))))

In [ ]:
write.csv(summary(counts(smallFiltered)), file = "/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/Test_{cDex}_Test.csv")

In [ ]:
CellTypes <- c("Type_1", "Type_2a", "Type_2x", "Endothelial", "Mesenchymal_Stem_Cell", "Macrophage", "Satellite_Cell", 
               "Neuromuscular_junction", "Neuronal", "Smooth_Muscle", "Muscle_Fiber_Mixed", "T_cell", "Adipocyte")
reFrame <- read.table("/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/dbgap/molecular/sn_muscle_nucleus_cluster_sample_mapping.tsv", sep = "\t", header = TRUE)

for (cDex in CellTypes) {
    
    print(glue("Loading and filtering {cDex} data..."))
    # Load ATAC
    small <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/{cDex}_FUSIONsnATAC_sce.rds"))
    smallF <- small[,small$sex == "F"]
    smallM <- small[,small$sex == "M"]
    keepRows <- rowSums(counts(small) >= 1) >= ncol(counts(small))*0.1
    small <- small[keepRows,]
    keepRowsF <- rowSums(counts(smallF) >= 1) >= ncol(counts(smallF))*0.1
    smallF <- smallF[keepRowsF,]
    keepRowsM <- rowSums(counts(smallM) >= 1) >= ncol(counts(smallM))*0.1
    smallM <- smallM[keepRowsM,]
    
    tempNucleiATAC <- dim(small)[2]
    
    # Load RNA
    sca <- readRDS(file = glue("/nfs/turbo/umms-scjp/kgmoo/data/rna_sample_counts/{cDex}_rna_postDecontx_postArushiQC_PostTPM_1in10nTotal_CDR_sca.rds"))
    scaF <- readRDS(file = glue("/nfs/turbo/umms-scjp/kgmoo/data/rna_sample_counts/{cDex}_rna_postDecontx_postArushiQC_PostTPM_1in10nFemale_CDR_sca.rds"))
    scaM <- readRDS(file = glue("/nfs/turbo/umms-scjp/kgmoo/data/rna_sample_counts/{cDex}_rna_postDecontx_postArushiQC_PostTPM_1in10nMale_CDR_sca.rds"))
    tempNucleiRNA <- dim(sca)[2]
    
    tDex <- "CleanUrFuckingData"
    
    if(cDex == "Mesenchymal_Stem_Cell") {
            tDex <- "FAP"
    } else if (cDex == "Neuromuscular_junction") {
            tDex <- "NMJ"
    } else {
            tDex <- cDex
    }
    
    tempreFrame <- reFrame[reFrame$coarse_cluster_name == glue("{tDex}"),]
    
    print(glue("ATAC Starting: {tempNucleiATAC} nuclei"))
    smaller <- small[,small$index %in% tempreFrame[tempreFrame$modality == "atac",]$index]
    tempNucleiATAC <- dim(smaller)[2]
    print(glue("ATAC Complete: {tempNucleiATAC} nuclei remain"))
    smallerF <- smallF[,smallF$index %in% tempreFrame[tempreFrame$modality == "atac",]$index]
    smallerM <- smallM[,smallM$index %in% tempreFrame[tempreFrame$modality == "atac",]$index]

    
    print(glue("RNA Starting: {tempNucleiRNA} nuclei"))
    scaler <- sca[,sca$id %in% tempreFrame[tempreFrame$modality == "rna",]$index]
    tempNucleiRNA <- dim(scaler)[2]
    print(glue("RNA Complete: {tempNucleiRNA} nuclei"))
    scalerF <- scaF[,scaF$id %in% tempreFrame[tempreFrame$modality == "rna",]$index]
    scalerM <- scaM[,scaM$id %in% tempreFrame[tempreFrame$modality == "rna",]$index]
    
    
    tempATACFrame <- data.frame(as.matrix(counts(smaller)))
    tempATACFrameF <- data.frame(as.matrix(counts(smallerF)))
    tempATACFrameM <- data.frame(as.matrix(counts(smallerM)))

    tempRNAFrame <- data.frame(as.matrix(counts(scaler)))
    tempRNAFrameF <- data.frame(as.matrix(counts(scalerF)))
    tempRNAFrameM <- data.frame(as.matrix(counts(scalerM)))

    #write.table(tempATACFrame, glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/snCountsMatrix_1ct10pct/FUSION_sampleTotal_tissueSM_nuclei{tDex}_atac1ct10pctN.tsv"), sep = "\t")
    #write.table(tempATACFrameF, glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/snCountsMatrix_1ct10pct/FUSION_sampleFemale_tissueSM_nuclei{tDex}_atac1ct10pctN.tsv"), sep = "\t")
    #write.table(tempATACFrameM, glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/snCountsMatrix_1ct10pct/FUSION_sampleMale_tissueSM_nuclei{tDex}_atac1ct10pctN.tsv"), sep = "\t")
    
    #write.table(tempRNAFrame, glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/snCountsMatrix_1ct10pct/FUSION_sampleTotal_tissueSM_nuclei{tDex}_rna1ct10pctN.tsv"), sep = "\t")
    #write.table(tempRNAFrameF, glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/snCountsMatrix_1ct10pct/FUSION_sampleFemale_tissueSM_nuclei{tDex}_rna1ct10pctN.tsv"), sep = "\t")
    #write.table(tempRNAFrameM, glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/snCountsMatrix_1ct10pct/FUSION_sampleMale_tissueSM_nuclei{tDex}_rna1ct10pctN.tsv"), sep = "\t")
}

In [3]:
cDex <- "Mesenchymal_Stem_Cell"   
reFrame <- read.table("/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/dbgap/molecular/sn_muscle_nucleus_cluster_sample_mapping.tsv", sep = "\t", header = TRUE)
print(glue("Loading and filtering {cDex} data..."))
    # Load ATAC
    small <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/{cDex}_FUSIONsnATAC_sce.rds"))
    keepRows <- rowSums(counts(small) >= 1) >= ncol(counts(small))*0.1
    small <- small[keepRows,]
    tempNucleiATAC <- dim(small)[2]
    
    # Load RNA
    sca <- readRDS(file = glue("/nfs/turbo/umms-scjp/kgmoo/data/rna_sample_counts/{cDex}_rna_postDecontx_postArushiQC_PostTPM_1in10nTotal_CDR_sca.rds"))
    tempNucleiRNA <- dim(sca)[2]
    
    tDex <- "CleanUrFuckingData"
    
    if(cDex == "Mesenchymal_Stem_Cell") {
            tDex <- "FAP"
    } else if (cDex == "Neuromuscular_junction") {
            tDex <- "NMJ"
    } else {
            tDex <- cDex
    }
    
    tempreFrame <- reFrame[reFrame$coarse_cluster_name == glue("{tDex}"),]

Loading and filtering Mesenchymal_Stem_Cell data...


In [10]:
length(unique(tempreFrame$SAMPLE_ID))
length(unique(tempreFrame$library))

head(tempreFrame)
head(small$index)
head(tempreFrame[tempreFrame$modality == "atac",]$index)

head(sca$id)
head(tempreFrame[tempreFrame$modality == "rna",]$index)

[1] 264

[1] 80

,index,UMAP_1,UMAP_2,modality,library,barcode,batch,SAMPLE_ID,hqaa_umi,fraction_mitochondrial,tss_enrichment,coarse_cluster_name
,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<chr>
8,rna.499.NM.1.AAAGGATGTGAGCTCC,-3.948268,-16.39488,rna,499-NM-1,AAAGGATGTGAGCTCC,499-NM,12086,5199,0.001845775,NA,FAP
20,rna.499.NM.1.AAAGTCCTCCTGTTGC,-4.801783,-16.66650,rna,499-NM-1,AAAGTCCTCCTGTTGC,499-NM,32073,2542,0.001015623,NA,FAP
72,rna.499.NM.1.AACTTCTTCCGATGCG,-9.047566,-16.34485,rna,499-NM-1,AACTTCTTCCGATGCG,499-NM,12011,6762,0.000447595,NA,FAP
81,rna.499.NM.1.AAGATAGGTCTGTGAT,-4.518937,-13.73997,rna,499-NM-1,AAGATAGGTCTGTGAT,499-NM,12099,6769,0.001496646,NA,FAP
89,rna.499.NM.1.AAGCGAGGTAGTGCGA,-7.020303,-16.47984,rna,499-NM-1,AAGCGAGGTAGTGCGA,499-NM,32054,3044,0.001601464,NA,FAP
99,rna.499.NM.1.AAGTACCCATTGCTTT,-8.789152,-15.85581,rna,499-NM-1,AAGTACCCATTGCTTT,499-NM,32052,3432,0.001337857,NA,FAP


[1] "atac.499.NM.1.AAACGAATCCAGAGAG" "atac.499.NM.1.AAACTCGCAGGAGCAT"
[3] "atac.499.NM.1.AAAGGATAGGGTTCCC" "atac.499.NM.1.AAATGCCGTGCACATT"
[5] "atac.499.NM.1.AAATGCCTCGCTCTAC" "atac.499.NM.1.AACATCGTCAGTCACA"

[1] "atac.499.NM.1.AAACGAATCCAGAGAG" "atac.499.NM.1.AAACTCGCAGGAGCAT"
[3] "atac.499.NM.1.AAAGGATAGGGTTCCC" "atac.499.NM.1.AAATGCCGTGCACATT"
[5] "atac.499.NM.1.AAATGCCTCGCTCTAC" "atac.499.NM.1.AACATCGTCAGTCACA"

[1] "rna.499.NM.1.AAAGGATGTGAGCTCC" "rna.499.NM.1.AAAGTCCTCCTGTTGC"
[3] "rna.499.NM.1.AACCTGAGTACATTGC" "rna.499.NM.1.AACTTCTTCCGATGCG"
[5] "rna.499.NM.1.AAGATAGGTCTGTGAT" "rna.499.NM.1.AAGCGAGGTAGTGCGA"

[1] "rna.499.NM.1.AAAGGATGTGAGCTCC" "rna.499.NM.1.AAAGTCCTCCTGTTGC"
[3] "rna.499.NM.1.AACTTCTTCCGATGCG" "rna.499.NM.1.AAGATAGGTCTGTGAT"
[5] "rna.499.NM.1.AAGCGAGGTAGTGCGA" "rna.499.NM.1.AAGTACCCATTGCTTT"

In [12]:
print(glue("ATAC Starting: {tempNucleiATAC} nuclei"))
smaller <- small[,small$index %in% tempreFrame[tempreFrame$modality == "atac",]$index]
tempNucleiATAC <- dim(smaller)[2]
print(glue("ATAC Complete: {tempNucleiATAC} nuclei remain"))

print(glue("RNA Starting: {tempNucleiRNA} nuclei"))
scaler <- sca[,sca$id %in% tempreFrame[tempreFrame$modality == "rna",]$index]
tempNucleiRNA <- dim(scaler)[2]
print(glue("RNA Complete: {tempNucleiRNA} nuclei"))

ATAC Starting: 13439 nuclei
ATAC Complete: 12259 nuclei remain
RNA Starting: 8439 nuclei
RNA Complete: 7673 nuclei
